In [1]:
!pip install torch_geometric 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.6 MB/s  0:00:00


In [2]:

# ================================================================
# END-TO-END GATv2-GRU EXPERIMENT
# Three seeds | Frozen baseline vs joint fine-tuning
# ================================================================

import os, time, copy, random, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv

# -------------------- CONFIGURATION --------------------

SEEDS = [42, 123, 2023]
MAX_EPOCHS = 100
PATIENCE = 15
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-5
DROPOUT = 0.2
HIDDEN_DIM = 128
NODE_BATCH_SIZE = 512

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = (
    "/kaggle/input/datasets/drrajgauravmishra/"
    "spatiotemporal-gnn-kaggle-data/"
    "Spatiotemporal_GNN_Kaggle_Data"
)

SPATIAL_DIR = (
    "/kaggle/input/notebooks/drrajgauravmishra/"
    "03-spatial-encoder-comparison-gcn-gat-gatv2-ipynb/"
    "spatial_encoder_results/checkpoints"
)

TEMPORAL_DIR = (
    "/kaggle/input/datasets/drrajgauravmishra/checkpoints"
)

OUTPUT_DIR = "/kaggle/working/end_to_end_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Device:", DEVICE)
print("Seeds:", SEEDS)

if DEVICE.type != "cuda":
    print("WARNING: GPU unavailable. Training may take longer.")

# -------------------- REPRODUCIBILITY --------------------

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# -------------------- LOAD DATA --------------------

train_data = np.load(os.path.join(DATA_DIR, "training_annual.npz"))
val_data = np.load(os.path.join(DATA_DIR, "validation_annual.npz"))

X_train = train_data["X"].astype(np.float32)
X_val = val_data["X"].astype(np.float32)

train_indices = train_data["input_indices"].astype(int)
val_indices = val_data["input_indices"].astype(int)

Y_train = train_data["Y"]
Y_val = val_data["Y"]

M_train = train_data["Y_mask"].astype(bool)
M_val = val_data["Y_mask"].astype(bool)

# Training-only target normalization
target_mean = np.zeros(2, dtype=np.float32)
target_std = np.zeros(2, dtype=np.float32)

for j in range(2):
    values = Y_train[:, :, j][M_train[:, :, j]]
    target_mean[j] = values.mean()
    target_std[j] = values.std()
    if target_std[j] == 0:
        target_std[j] = 1.0

def normalize_targets(Y, mask):
    result = np.zeros_like(Y, dtype=np.float32)
    for j in range(2):
        valid = mask[:, :, j]
        result[:, :, j][valid] = (
            Y[:, :, j][valid] - target_mean[j]
        ) / target_std[j]
    return result

Y_train = normalize_targets(Y_train, M_train)
Y_val = normalize_targets(Y_val, M_val)

Y_train = torch.tensor(Y_train, dtype=torch.float32, device=DEVICE)
Y_val = torch.tensor(Y_val, dtype=torch.float32, device=DEVICE)

M_train = torch.tensor(M_train, dtype=torch.bool, device=DEVICE)
M_val = torch.tensor(M_val, dtype=torch.bool, device=DEVICE)

# Reconstruct unique annual input snapshots from train and validation.
annual_map = {}

for X, indices in [(X_train, train_indices), (X_val, val_indices)]:
    for w in range(X.shape[0]):
        for t in range(X.shape[1]):
            year = int(indices[w, t])
            snapshot = X[w, t, :, :2]

            if year in annual_map:
                if not np.allclose(
                    annual_map[year], snapshot,
                    atol=1e-6, rtol=0, equal_nan=True
                ):
                    raise ValueError(f"Inconsistent annual input: {year}")
            else:
                annual_map[year] = snapshot.copy()

train_years = sorted(set(train_indices.flatten().tolist()))
val_years = sorted(set(val_indices.flatten().tolist()))

print("Training windows:", len(X_train))
print("Validation windows:", len(X_val))
print("Annual input snapshots:", len(annual_map))

# -------------------- LOAD STATIC GRAPH --------------------

graph_path = os.path.join(DATA_DIR, "static_spatial_graph_verified.pt")
graph = torch.load(graph_path, map_location="cpu", weights_only=False)

edge_index = graph.edge_index.to(DEVICE)

assert graph.num_nodes == 5040
assert edge_index.shape == (2, 40320)

print("Graph loaded:", graph.num_nodes, "nodes;",
      edge_index.shape[1], "edges")

# -------------------- MODEL DEFINITIONS --------------------

class GATv2Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = GATv2Conv(
            2, 64, heads=4, concat=True,
            dropout=0.0, add_self_loops=False
        )

        self.conv2 = GATv2Conv(
            256, 2, heads=1, concat=False,
            dropout=0.0, add_self_loops=False
        )

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        return h


class GATv2GRUForecaster(nn.Module):
    def __init__(
        self, input_dim=256, hidden_dim=128,
        num_layers=1, output_dim=2, dropout=0.2
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        batch, steps, nodes, features = x.shape

        x = x.permute(0, 2, 1, 3)
        x = x.reshape(batch * nodes, steps, features)

        _, hidden = self.gru(x)
        output = self.fc(self.dropout(hidden[-1]))

        return output.reshape(batch, nodes, 2)


# -------------------- CHECKPOINT RECOVERY --------------------

def recover_temporal_checkpoint(seed):
    name = f"GATv2_GRU_Seed{seed}"
    source = os.path.join(TEMPORAL_DIR, name, name)
    destination = os.path.join(OUTPUT_DIR, name + ".pt")

    if not os.path.isdir(source):
        raise FileNotFoundError(source)

    parent = os.path.dirname(source)

    with zipfile.ZipFile(
        destination, "w", zipfile.ZIP_STORED
    ) as archive:
        for root, dirs, files in os.walk(source):
            for filename in files:
                full = os.path.join(root, filename)
                relative = os.path.relpath(full, parent)
                archive.write(full, arcname=relative)

    return torch.load(
        destination, map_location="cpu", weights_only=False
    )


# -------------------- EMBEDDING FUNCTIONS --------------------

def encode_years(encoder, years, use_grad):
    encoded = {}

    context = torch.enable_grad() if use_grad else torch.no_grad()

    with context:
        for year in years:
            x = torch.as_tensor(
                annual_map[int(year)],
                dtype=torch.float32,
                device=DEVICE
            )
            encoded[int(year)] = encoder(x, edge_index)

    return encoded


def make_sequence(encoded, indices, start, end):
    windows = []

    for row in indices:
        years = [
            encoded[int(year)][start:end]
            for year in row
        ]
        windows.append(torch.stack(years, dim=0))

    return torch.stack(windows, dim=0)


def evaluate_model(encoder, temporal_model, years, indices, targets, masks):
    encoder.eval()
    temporal_model.eval()

    encoded = encode_years(encoder, years, use_grad=False)

    squared_error = 0.0
    valid_count = 0
    nodes = targets.shape[1]

    with torch.no_grad():
        for start in range(0, nodes, NODE_BATCH_SIZE):
            end = min(start + NODE_BATCH_SIZE, nodes)

            sequence = make_sequence(encoded, indices, start, end)
            prediction = temporal_model(sequence)

            target = targets[:, start:end, :]
            mask = masks[:, start:end, :]

            if mask.any():
                squared_error += (
                    (prediction[mask] - target[mask]) ** 2
                ).sum().item()
                valid_count += mask.sum().item()

    del encoded

    return squared_error / valid_count


# -------------------- RUN ALL THREE SEEDS --------------------

all_results = []

for seed in SEEDS:

    print("\n" + "=" * 65)
    print(f"STARTING SEED {seed}")
    print("=" * 65)

    set_seed(seed)

    spatial_path = os.path.join(
        SPATIAL_DIR, f"GATv2_seed_{seed}.pt"
    )

    if not os.path.isfile(spatial_path):
        raise FileNotFoundError(spatial_path)

    spatial_ckpt = torch.load(
        spatial_path, map_location="cpu", weights_only=False
    )

    temporal_ckpt = recover_temporal_checkpoint(seed)

    # Verify target normalization against the original temporal model.
    assert np.allclose(
        target_mean,
        np.asarray(temporal_ckpt["target_mean"]).reshape(2),
        atol=1e-5
    )
    assert np.allclose(
        target_std,
        np.asarray(temporal_ckpt["target_std"]).reshape(2),
        atol=1e-5
    )

    assert temporal_ckpt["input_dim"] == 256
    assert temporal_ckpt["hidden_dim"] == HIDDEN_DIM

    encoder = GATv2Encoder().to(DEVICE)
    encoder.load_state_dict(
        spatial_ckpt["model_state_dict"], strict=True
    )

    temporal_model = GATv2GRUForecaster(
        input_dim=256,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT
    ).to(DEVICE)

    temporal_model.load_state_dict(
        temporal_ckpt["model_state_dict"], strict=True
    )

    # Frozen spatial encoder baseline.
    initial_val_loss = evaluate_model(
        encoder, temporal_model,
        val_years, val_indices, Y_val, M_val
    )

    print(f"Frozen encoder validation MSE: {initial_val_loss:.6f}")

    # Joint fine-tuning.
    optimizer = torch.optim.Adam(
        list(encoder.parameters()) +
        list(temporal_model.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    best_val_loss = float("inf")
    best_epoch = 0
    patience_counter = 0
    train_history = []
    val_history = []
    start_time = time.time()

    total_valid = int(M_train.sum().item())
    number_of_batches = (
        Y_train.shape[1] + NODE_BATCH_SIZE - 1
    ) // NODE_BATCH_SIZE

    for epoch in range(1, MAX_EPOCHS + 1):

        encoder.train()
        temporal_model.train()
        optimizer.zero_grad()

        train_embeddings = encode_years(
            encoder, train_years, use_grad=True
        )

        epoch_squared_error = 0.0
        last_gradient_norm = 0.0

        for batch_idx, start in enumerate(
            range(0, Y_train.shape[1], NODE_BATCH_SIZE)
        ):
            end = min(start + NODE_BATCH_SIZE, Y_train.shape[1])

            sequence = make_sequence(
                train_embeddings, train_indices, start, end
            )

            prediction = temporal_model(sequence)

            target = Y_train[:, start:end, :]
            mask = M_train[:, start:end, :]

            if mask.any():
                squared = (prediction[mask] - target[mask]) ** 2
                epoch_squared_error += squared.sum().item()

                loss = squared.sum() / total_valid

                loss.backward(
                    retain_graph=(batch_idx < number_of_batches - 1)
                )

        gradient = encoder.conv1.lin_l.weight.grad

        if gradient is None:
            raise RuntimeError("GATv2 did not receive forecasting gradients.")

        last_gradient_norm = gradient.norm().item()

        if not np.isfinite(last_gradient_norm) or last_gradient_norm == 0:
            raise RuntimeError("Invalid GATv2 forecasting gradient.")

        optimizer.step()
        del train_embeddings

        train_loss = epoch_squared_error / total_valid

        val_loss = evaluate_model(
            encoder, temporal_model,
            val_years, val_indices, Y_val, M_val
        )

        train_history.append(train_loss)
        val_history.append(val_loss)

        print(
            f"Seed {seed} | Epoch {epoch:03d} | "
            f"Train MSE {train_loss:.6f} | "
            f"Val MSE {val_loss:.6f} | "
            f"GAT gradient {last_gradient_norm:.3e}"
        )

        if val_loss < best_val_loss - 1e-7:
            best_val_loss = val_loss
            best_epoch = epoch
            patience_counter = 0

            best_encoder_state = {
                k: v.detach().cpu().clone()
                for k, v in encoder.state_dict().items()
            }

            best_temporal_state = {
                k: v.detach().cpu().clone()
                for k, v in temporal_model.state_dict().items()
            }

        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print("Early stopping.")
            break

    elapsed = time.time() - start_time

    # Restore best validation checkpoint.
    encoder.load_state_dict(best_encoder_state)
    temporal_model.load_state_dict(best_temporal_state)

    final_val_loss = evaluate_model(
        encoder, temporal_model,
        val_years, val_indices, Y_val, M_val
    )

    checkpoint_output = os.path.join(
        OUTPUT_DIR, f"end_to_end_seed_{seed}.pt"
    )

    torch.save({
        "seed": seed,
        "encoder_state_dict": best_encoder_state,
        "temporal_state_dict": best_temporal_state,
        "initial_frozen_val_mse": initial_val_loss,
        "best_val_mse": final_val_loss,
        "best_epoch": best_epoch,
        "train_history": train_history,
        "val_history": val_history,
        "training_time_seconds": elapsed,
        "target_mean": target_mean,
        "target_std": target_std,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "patience": PATIENCE
    }, checkpoint_output)

    improvement = (
        (initial_val_loss - final_val_loss)
        / initial_val_loss * 100
    )

    all_results.append({
        "Seed": seed,
        "Frozen_Val_MSE": initial_val_loss,
        "End_to_End_Val_MSE": final_val_loss,
        "Improvement_Percent": improvement,
        "Best_Epoch": best_epoch,
        "Training_Time_Min": elapsed / 60,
        "Checkpoint": checkpoint_output
    })

    print(f"\nSeed {seed} completed.")
    print(f"Frozen validation MSE: {initial_val_loss:.6f}")
    print(f"Fine-tuned validation MSE: {final_val_loss:.6f}")
    print(f"Improvement: {improvement:.2f}%")
    print(f"Best epoch: {best_epoch}")
    print(f"Time: {elapsed / 60:.2f} minutes")

    del encoder, temporal_model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# -------------------- FINAL SUMMARY --------------------

summary = pd.DataFrame(all_results)

summary_path = os.path.join(
    OUTPUT_DIR, "end_to_end_experiment_summary.csv"
)

summary.to_csv(summary_path, index=False)

print("\n" + "=" * 75)
print("FINAL END-TO-END EXPERIMENT SUMMARY")
print("=" * 75)

print(summary.to_string(index=False))

print("\nMean frozen validation MSE:",
      summary["Frozen_Val_MSE"].mean())

print("Mean end-to-end validation MSE:",
      summary["End_to_End_Val_MSE"].mean())

print("\nResults saved to:", OUTPUT_DIR)
print("Summary CSV:", summary_path)
print("\nEXPERIMENT COMPLETED.")


Device: cuda
Seeds: [42, 123, 2023]
Training windows: 8
Validation windows: 6
Annual input snapshots: 23
Graph loaded: 5040 nodes; 40320 edges

STARTING SEED 42
Frozen encoder validation MSE: 0.136783
Seed 42 | Epoch 001 | Train MSE 0.163307 | Val MSE 0.189725 | GAT gradient 1.196e-02
Seed 42 | Epoch 002 | Train MSE 0.197061 | Val MSE 0.140745 | GAT gradient 7.822e-02
Seed 42 | Epoch 003 | Train MSE 0.165810 | Val MSE 0.138333 | GAT gradient 2.692e-02
Seed 42 | Epoch 004 | Train MSE 0.176827 | Val MSE 0.141189 | GAT gradient 5.561e-02
Seed 42 | Epoch 005 | Train MSE 0.182316 | Val MSE 0.135508 | GAT gradient 5.332e-02
Seed 42 | Epoch 006 | Train MSE 0.172340 | Val MSE 0.134265 | GAT gradient 4.216e-02
Seed 42 | Epoch 007 | Train MSE 0.163294 | Val MSE 0.142739 | GAT gradient 3.289e-02
Seed 42 | Epoch 008 | Train MSE 0.164444 | Val MSE 0.153437 | GAT gradient 2.915e-02
Seed 42 | Epoch 009 | Train MSE 0.170095 | Val MSE 0.157080 | GAT gradient 3.258e-02
Seed 42 | Epoch 010 | Train MSE 0.

In [3]:
import os
import numpy as np

DATA_DIR = (
    "/kaggle/input/datasets/drrajgauravmishra/"
    "spatiotemporal-gnn-kaggle-data/"
    "Spatiotemporal_GNN_Kaggle_Data"
)

print("Available files:")
for f in sorted(os.listdir(DATA_DIR)):
    print(f)

print("\nNPZ dataset details:")
for f in sorted(os.listdir(DATA_DIR)):
    if f.endswith(".npz"):
        data = np.load(os.path.join(DATA_DIR, f))
        print(f"\n{f}")
        for key in data.files:
            print(key, data[key].shape, data[key].dtype)

Available files:
annual_forecasting_metadata.json
static_spatial_graph_verified.pt
testing_annual.npz
training_annual.npz
validation_annual.npz

NPZ dataset details:

testing_annual.npz
X (2, 10, 5040, 4) float32
Y (2, 5040, 2) float32
Y_mask (2, 5040, 2) bool
input_indices (2, 10) int64
target_indices (2,) int64
target_years (2,) int64

training_annual.npz
X (8, 10, 5040, 4) float32
Y (8, 5040, 2) float32
Y_mask (8, 5040, 2) bool
input_indices (8, 10) int64
target_indices (8,) int64
target_years (8,) int64

validation_annual.npz
X (6, 10, 5040, 4) float32
Y (6, 5040, 2) float32
Y_mask (6, 5040, 2) bool
input_indices (6, 10) int64
target_indices (6,) int64
target_years (6,) int64


In [4]:

import os
import json
import numpy as np

DATA_DIR = (
    "/kaggle/input/datasets/drrajgauravmishra/"
    "spatiotemporal-gnn-kaggle-data/"
    "Spatiotemporal_GNN_Kaggle_Data"
)

# Inspect metadata
metadata_path = os.path.join(
    DATA_DIR, "annual_forecasting_metadata.json"
)

with open(metadata_path, "r") as f:
    metadata = json.load(f)

print("DATASET METADATA")
print(json.dumps(metadata, indent=2))

# Inspect all three splits
for split in ["training", "validation", "testing"]:

    data = np.load(
        os.path.join(DATA_DIR, f"{split}_annual.npz")
    )

    print("\n" + "=" * 65)
    print(split.upper())
    print("=" * 65)

    print("Input indices:")
    print(data["input_indices"])

    print("Target years:")
    print(data["target_years"])

    print("Target indices:")
    print(data["target_indices"])

    print("Valid observations:")
    for j, variable in enumerate(
        ["Rainfall", "Maximum Temperature"]
    ):
        count = int(data["Y_mask"][:, :, j].sum())
        print(f"{variable}: {count}")

    print("Input years:", np.unique(data["input_indices"]))


DATASET METADATA
{
  "forecasting_task": "Annual one-year-ahead forecasting",
  "input_window_years": 10,
  "historical_period": "2000-2025",
  "training_target_years": [
    2010,
    2011,
    2012,
    2013,
    2014,
    2015,
    2016,
    2017
  ],
  "validation_target_years": [
    2018,
    2019,
    2020,
    2021,
    2022,
    2023
  ],
  "testing_target_years": [
    2024,
    2025
  ],
  "number_of_nodes": 5040,
  "number_of_input_features": 4,
  "input_features": [
    "Normalized rainfall",
    "Normalized maximum temperature",
    "Rainfall missingness indicator",
    "Temperature missingness indicator"
  ],
  "target_variables": [
    "Rainfall (mm/day)",
    "Maximum temperature (degree_Celsius)"
  ],
  "annual_coverage_threshold_percent": 90.0,
  "imputation_method": "Training-only node-wise mean; global training mean fallback",
  "normalization": {
    "rainfall_mean": 3.048717737197876,
    "rainfall_std": 2.088418483734131,
    "temperature_mean": 31.2358303070068

In [5]:

test_data = np.load(
    os.path.join(DATA_DIR, "testing_annual.npz")
)

X_test = test_data["X"].astype(np.float32)
test_indices = test_data["input_indices"].astype(int)

Y_test_raw = test_data["Y"].astype(np.float32)
M_test = test_data["Y_mask"].astype(bool)

# Extend annual input snapshots with test inputs.
for w in range(X_test.shape[0]):
    for t in range(X_test.shape[1]):

        year = int(test_indices[w, t])
        snapshot = X_test[w, t, :, :2]

        if year in annual_map:
            assert np.allclose(
                annual_map[year],
                snapshot,
                atol=1e-6,
                rtol=0,
                equal_nan=True
            ), f"Input mismatch for year index {year}"

        else:
            annual_map[year] = snapshot.copy()

print("Available annual snapshots:",
      min(annual_map), "to", max(annual_map))

print("Test target years:",
      test_data["target_years"])

assert all(
    int(year) in annual_map
    for year in test_indices.flatten()
)

print("All test input snapshots available.")


Available annual snapshots: 0 to 24
Test target years: [2024 2025]
All test input snapshots available.


In [6]:
def forward(self, x, edge_index):
    h = F.relu(self.conv1(x, edge_index))
    return h

In [7]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv

class GATv2Encoder(nn.Module):
    def __init__(self, in_channels=2, hidden_channels=64,
                 out_channels=2, heads=4, dropout=0.0):
        super().__init__()

        self.conv1 = GATv2Conv(
            in_channels, hidden_channels,
            heads=heads, concat=True,
            dropout=dropout, add_self_loops=False
        )

        self.conv2 = GATv2Conv(
            hidden_channels * heads, out_channels,
            heads=1, concat=False,
            dropout=dropout, add_self_loops=False
        )

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        reconstruction = self.conv2(h, edge_index)
        return reconstruction, h


class GATv2ForecastEncoder(nn.Module):
    def __init__(self, in_channels=2, hidden_channels=64,
                 heads=4, dropout=0.0):
        super().__init__()

        self.conv1 = GATv2Conv(
            in_channels, hidden_channels,
            heads=heads, concat=True,
            dropout=dropout, add_self_loops=False
        )

    def forward(self, x, edge_index):
        return F.relu(self.conv1(x, edge_index))


# Inspect one checkpoint first
checkpoint_path = (
    "/kaggle/input/notebooks/drrajgauravmishra/"
    "03-spatial-encoder-comparison-gcn-gat-gatv2-ipynb/"
    "spatial_encoder_results/checkpoints/GATv2_seed_42.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", checkpoint.keys())


Checkpoint type: <class 'dict'>
Checkpoint keys: dict_keys(['model_name', 'seed', 'model_state_dict', 'hidden_dim', 'learning_rate', 'epochs', 'training_loss_history'])


In [8]:

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv

# ------------------------------------------------
# Original spatial encoder
# ------------------------------------------------

class OriginalGATv2Encoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = GATv2Conv(
            2, 64, heads=4,
            concat=True, dropout=0.0,
            add_self_loops=False
        )

        self.conv2 = GATv2Conv(
            256, 2, heads=1,
            concat=False, dropout=0.0,
            add_self_loops=False
        )

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        reconstruction = self.conv2(h, edge_index)
        return reconstruction, h


# ------------------------------------------------
# Forecasting encoder
# ------------------------------------------------

class ForecastGATv2Encoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = GATv2Conv(
            2, 64, heads=4,
            concat=True, dropout=0.0,
            add_self_loops=False
        )

    def forward(self, x, edge_index):
        return F.relu(self.conv1(x, edge_index))


# ------------------------------------------------
# Compare all three pretrained checkpoints
# ------------------------------------------------

SPATIAL_DIR = (
    "/kaggle/input/notebooks/drrajgauravmishra/"
    "03-spatial-encoder-comparison-gcn-gat-gatv2-ipynb/"
    "spatial_encoder_results/checkpoints"
)

# Use one identical annual snapshot for all seeds.
test_input = torch.as_tensor(
    annual_map[0],
    dtype=torch.float32,
    device=DEVICE
)

results = []

for seed in [42, 123, 2023]:

    print("\nChecking seed:", seed)

    checkpoint_path = os.path.join(
        SPATIAL_DIR, f"GATv2_seed_{seed}.pt"
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    state = checkpoint["model_state_dict"]

    print("Checkpoint seed:", checkpoint["seed"])
    print("Hidden dimension:", checkpoint["hidden_dim"])
    print("State dictionary keys:", list(state.keys()))

    # Original encoder
    original = OriginalGATv2Encoder().to(DEVICE)
    original.load_state_dict(state, strict=True)

    # Forecasting encoder: transfer conv1 only
    forecast = ForecastGATv2Encoder().to(DEVICE)

    conv1_state = {
        k.replace("conv1.", "", 1): v
        for k, v in state.items()
        if k.startswith("conv1.")
    }

    forecast.conv1.load_state_dict(
        conv1_state, strict=True
    )

    original.eval()
    forecast.eval()

    with torch.no_grad():

        reconstruction, original_embedding = original(
            test_input, edge_index
        )

        forecast_embedding = forecast(
            test_input, edge_index
        )

    difference = (
        original_embedding - forecast_embedding
    ).abs()

    max_difference = difference.max().item()
    mean_difference = difference.mean().item()

    passed = torch.allclose(
        original_embedding,
        forecast_embedding,
        atol=1e-6,
        rtol=1e-5
    )

    results.append({
        "Seed": seed,
        "Max_Absolute_Difference": max_difference,
        "Mean_Absolute_Difference": mean_difference,
        "Passed": passed
    })

    print("Original embedding:", original_embedding.shape)
    print("Forecast embedding:", forecast_embedding.shape)
    print("Reconstruction:", reconstruction.shape)
    print("Maximum difference:", max_difference)
    print("Mean difference:", mean_difference)
    print("Equivalence:", "PASS" if passed else "FAIL")

    assert passed, f"Embedding mismatch for seed {seed}"

    del original, forecast

print("\n" + "=" * 65)
print("EMBEDDING EQUIVALENCE SUMMARY")
print("=" * 65)

import pandas as pd

equivalence_df = pd.DataFrame(results)
print(equivalence_df.to_string(index=False))

assert equivalence_df["Passed"].all()

print("\nALL THREE SEEDS PASSED.")



Checking seed: 42
Checkpoint seed: 42
Hidden dimension: 64
State dictionary keys: ['conv1.att', 'conv1.bias', 'conv1.lin_l.weight', 'conv1.lin_l.bias', 'conv1.lin_r.weight', 'conv1.lin_r.bias', 'conv2.att', 'conv2.bias', 'conv2.lin_l.weight', 'conv2.lin_l.bias', 'conv2.lin_r.weight', 'conv2.lin_r.bias']
Original embedding: torch.Size([5040, 256])
Forecast embedding: torch.Size([5040, 256])
Reconstruction: torch.Size([5040, 2])
Maximum difference: 3.5762786865234375e-07
Mean difference: 4.882145798745796e-09
Equivalence: PASS

Checking seed: 123
Checkpoint seed: 123
Hidden dimension: 64
State dictionary keys: ['conv1.att', 'conv1.bias', 'conv1.lin_l.weight', 'conv1.lin_l.bias', 'conv1.lin_r.weight', 'conv1.lin_r.bias', 'conv2.att', 'conv2.bias', 'conv2.lin_l.weight', 'conv2.lin_l.bias', 'conv2.lin_r.weight', 'conv2.lin_r.bias']
Original embedding: torch.Size([5040, 256])
Forecast embedding: torch.Size([5040, 256])
Reconstruction: torch.Size([5040, 2])
Maximum difference: 4.768371582031

In [9]:

# ================================================================
# INDEPENDENT TEST EVALUATION
# Frozen GATv2-GRU vs End-to-End GATv2-GRU
# Three seeds | Test years 2024 and 2025
# ================================================================

import os
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GATv2Conv

# ------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------

SEEDS = [42, 123, 2023]
NODE_BATCH_SIZE = 512
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DATA_DIR = (
    "/kaggle/input/datasets/drrajgauravmishra/"
    "spatiotemporal-gnn-kaggle-data/"
    "Spatiotemporal_GNN_Kaggle_Data"
)

SPATIAL_DIR = (
    "/kaggle/input/notebooks/drrajgauravmishra/"
    "03-spatial-encoder-comparison-gcn-gat-gatv2-ipynb/"
    "spatial_encoder_results/checkpoints"
)

TEMPORAL_DIR = (
    "/kaggle/input/datasets/drrajgauravmishra/checkpoints"
)

E2E_DIR = "/kaggle/working/end_to_end_results"

OUTPUT_DIR = "/kaggle/working/final_test_evaluation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Device:", DEVICE)
print("Seeds:", SEEDS)

# ------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------

train_data = np.load(
    os.path.join(DATA_DIR, "training_annual.npz")
)

val_data = np.load(
    os.path.join(DATA_DIR, "validation_annual.npz")
)

test_data = np.load(
    os.path.join(DATA_DIR, "testing_annual.npz")
)

X_train = train_data["X"].astype(np.float32)
X_val = val_data["X"].astype(np.float32)
X_test = test_data["X"].astype(np.float32)

train_indices = train_data["input_indices"].astype(int)
val_indices = val_data["input_indices"].astype(int)
test_indices = test_data["input_indices"].astype(int)

Y_test_raw = test_data["Y"].astype(np.float32)
M_test = test_data["Y_mask"].astype(bool)
test_years = test_data["target_years"].astype(int)

assert test_years.tolist() == [2024, 2025]
assert X_test.shape == (2, 10, 5040, 4)

print("\nTest years:", test_years)
print("Test targets:", Y_test_raw.shape)
print("Test masks:", M_test.shape)

# ------------------------------------------------
# 3. VERIFY TRAINING-ONLY NORMALIZATION
# ------------------------------------------------

Y_train_raw = train_data["Y"].astype(np.float32)
M_train = train_data["Y_mask"].astype(bool)

target_mean = np.zeros(2, dtype=np.float32)
target_std = np.zeros(2, dtype=np.float32)

for j in range(2):

    values = Y_train_raw[:, :, j][M_train[:, :, j]]

    target_mean[j] = values.mean()
    target_std[j] = values.std()

    if target_std[j] == 0:
        target_std[j] = 1.0

print("\nTraining target statistics:")
print("Mean:", target_mean)
print("Std :", target_std)

# ------------------------------------------------
# 4. RECONSTRUCT ALL ANNUAL INPUT SNAPSHOTS
# ------------------------------------------------

annual_map = {}

for X, indices in [
    (X_train, train_indices),
    (X_val, val_indices),
    (X_test, test_indices)
]:

    for w in range(X.shape[0]):
        for t in range(X.shape[1]):

            year = int(indices[w, t])
            snapshot = X[w, t, :, :2]

            if year in annual_map:

                assert np.allclose(
                    annual_map[year],
                    snapshot,
                    atol=1e-6,
                    rtol=0,
                    equal_nan=True
                ), f"Inconsistent snapshot: {year}"

            else:
                annual_map[year] = snapshot.copy()

print("\nAnnual snapshots:",
      min(annual_map), "to", max(annual_map))

# ------------------------------------------------
# 5. LOAD VERIFIED STATIC GRAPH
# ------------------------------------------------

graph_path = os.path.join(
    DATA_DIR,
    "static_spatial_graph_verified.pt"
)

graph = torch.load(
    graph_path,
    map_location="cpu",
    weights_only=False
)

edge_index = graph.edge_index.to(DEVICE)

assert graph.num_nodes == 5040
assert edge_index.shape == (2, 40320)

print("Graph:", graph.num_nodes, "nodes")

# ------------------------------------------------
# 6. MODEL DEFINITIONS
# Identical to the original experiment
# ------------------------------------------------

class GATv2Encoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = GATv2Conv(
            2, 64, heads=4,
            concat=True,
            dropout=0.0,
            add_self_loops=False
        )

        self.conv2 = GATv2Conv(
            256, 2, heads=1,
            concat=False,
            dropout=0.0,
            add_self_loops=False
        )

    def forward(self, x, edge_index):

        h = F.relu(
            self.conv1(x, edge_index)
        )

        return h


class GATv2GRUForecaster(nn.Module):

    def __init__(
        self,
        input_dim=256,
        hidden_dim=128,
        num_layers=1,
        output_dim=2,
        dropout=0.2
    ):

        super().__init__()

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_dim, output_dim
        )

    def forward(self, x):

        batch, steps, nodes, features = x.shape

        x = x.permute(0, 2, 1, 3)

        x = x.reshape(
            batch * nodes, steps, features
        )

        _, hidden = self.gru(x)

        output = self.fc(
            self.dropout(hidden[-1])
        )

        return output.reshape(batch, nodes, 2)


# ------------------------------------------------
# 7. RECOVER ORIGINAL TEMPORAL CHECKPOINTS
# ------------------------------------------------

def recover_temporal_checkpoint(seed):

    name = f"GATv2_GRU_Seed{seed}"

    source = os.path.join(
        TEMPORAL_DIR, name, name
    )

    destination = os.path.join(
        OUTPUT_DIR, name + ".pt"
    )

    if not os.path.isdir(source):
        raise FileNotFoundError(source)

    parent = os.path.dirname(source)

    with zipfile.ZipFile(
        destination, "w", zipfile.ZIP_STORED
    ) as archive:

        for root, dirs, files in os.walk(source):

            for filename in files:

                full = os.path.join(root, filename)

                relative = os.path.relpath(
                    full, parent
                )

                archive.write(
                    full, arcname=relative
                )

    return torch.load(
        destination,
        map_location="cpu",
        weights_only=False
    )


# ------------------------------------------------
# 8. ENCODING AND SEQUENCE GENERATION
# ------------------------------------------------

def encode_years(encoder, years):

    encoded = {}

    encoder.eval()

    with torch.no_grad():

        for year in years:

            x = torch.as_tensor(
                annual_map[int(year)],
                dtype=torch.float32,
                device=DEVICE
            )

            encoded[int(year)] = encoder(
                x, edge_index
            )

    return encoded


def make_sequence(encoded, indices, start, end):

    windows = []

    for row in indices:

        years = [
            encoded[int(year)][start:end]
            for year in row
        ]

        windows.append(
            torch.stack(years, dim=0)
        )

    return torch.stack(windows, dim=0)


# ------------------------------------------------
# 9. TEST PREDICTION FUNCTION
# ------------------------------------------------

def generate_predictions(
    encoder,
    temporal_model,
    indices,
    targets
):

    encoder.eval()
    temporal_model.eval()

    required_years = sorted(
        set(indices.flatten().tolist())
    )

    encoded = encode_years(
        encoder, required_years
    )

    predictions_normalized = np.zeros_like(
        targets, dtype=np.float32
    )

    nodes = targets.shape[1]

    with torch.no_grad():

        for start in range(
            0, nodes, NODE_BATCH_SIZE
        ):

            end = min(
                start + NODE_BATCH_SIZE,
                nodes
            )

            sequence = make_sequence(
                encoded, indices, start, end
            )

            prediction = temporal_model(sequence)

            predictions_normalized[
                :, start:end, :
            ] = prediction.cpu().numpy()

    del encoded

    # Convert predictions back to physical units.
    predictions_physical = (
        predictions_normalized *
        target_std.reshape(1, 1, 2)
        +
        target_mean.reshape(1, 1, 2)
    )

    return predictions_physical


# ------------------------------------------------
# 10. METRIC CALCULATION
# ------------------------------------------------

def calculate_metrics(y_true, y_pred):

    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    error = y_pred - y_true

    mse = np.mean(error ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(error))

    ss_res = np.sum(error ** 2)
    ss_tot = np.sum(
        (y_true - y_true.mean()) ** 2
    )

    r2 = (
        1.0 - ss_res / ss_tot
        if ss_tot > 0 else np.nan
    )

    return {
        "N": len(y_true),
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    }


def evaluate_metrics(
    seed,
    model_name,
    predictions
):

    records = []

    variable_names = [
        "Rainfall",
        "Maximum_Temperature"
    ]

    # Individual years
    for i, year in enumerate(test_years):

        for j, variable in enumerate(variable_names):

            mask = M_test[i, :, j]

            metrics = calculate_metrics(
                Y_test_raw[i, :, j][mask],
                predictions[i, :, j][mask]
            )

            records.append({
                "Seed": seed,
                "Model": model_name,
                "Year": int(year),
                "Variable": variable,
                **metrics
            })

    # Pooled results across both test years
    for j, variable in enumerate(variable_names):

        mask = M_test[:, :, j]

        metrics = calculate_metrics(
            Y_test_raw[:, :, j][mask],
            predictions[:, :, j][mask]
        )

        records.append({
            "Seed": seed,
            "Model": model_name,
            "Year": "Pooled_2024_2025",
            "Variable": variable,
            **metrics
        })

    return records


# ------------------------------------------------
# 11. RUN FROZEN VS END-TO-END TEST EVALUATION
# ------------------------------------------------

all_metrics = []

for seed in SEEDS:

    print("\n" + "=" * 75)
    print("EVALUATING SEED:", seed)
    print("=" * 75)

    # Load original spatial checkpoint
    spatial_path = os.path.join(
        SPATIAL_DIR,
        f"GATv2_seed_{seed}.pt"
    )

    spatial_ckpt = torch.load(
        spatial_path,
        map_location="cpu",
        weights_only=False
    )

    # Load original temporal checkpoint
    temporal_ckpt = recover_temporal_checkpoint(seed)

    # Verify target normalization
    checkpoint_mean = np.asarray(
        temporal_ckpt["target_mean"]
    ).reshape(2)

    checkpoint_std = np.asarray(
        temporal_ckpt["target_std"]
    ).reshape(2)

    assert np.allclose(
        checkpoint_mean, target_mean, atol=1e-5
    )

    assert np.allclose(
        checkpoint_std, target_std, atol=1e-5
    )

    assert temporal_ckpt["input_dim"] == 256
    assert temporal_ckpt["hidden_dim"] == 128

    # --------------------------------------------
    # A. FROZEN BASELINE
    # --------------------------------------------

    frozen_encoder = GATv2Encoder().to(DEVICE)

    frozen_encoder.load_state_dict(
        spatial_ckpt["model_state_dict"],
        strict=True
    )

    frozen_temporal = GATv2GRUForecaster(
        input_dim=256,
        hidden_dim=128,
        dropout=0.2
    ).to(DEVICE)

    frozen_temporal.load_state_dict(
        temporal_ckpt["model_state_dict"],
        strict=True
    )

    print("\nGenerating Frozen predictions...")

    frozen_predictions = generate_predictions(
        frozen_encoder,
        frozen_temporal,
        test_indices,
        Y_test_raw
    )

    frozen_metrics = evaluate_metrics(
        seed,
        "Frozen",
        frozen_predictions
    )

    all_metrics.extend(frozen_metrics)

    print("Frozen evaluation completed.")

    # --------------------------------------------
    # B. END-TO-END MODEL
    # --------------------------------------------

    e2e_path = os.path.join(
        E2E_DIR,
        f"end_to_end_seed_{seed}.pt"
    )

    if not os.path.isfile(e2e_path):
        raise FileNotFoundError(
            f"End-to-End checkpoint missing: {e2e_path}"
        )

    e2e_ckpt = torch.load(
        e2e_path,
        map_location="cpu",
        weights_only=False
    )

    assert e2e_ckpt["seed"] == seed

    e2e_encoder = GATv2Encoder().to(DEVICE)

    e2e_encoder.load_state_dict(
        e2e_ckpt["encoder_state_dict"],
        strict=True
    )

    e2e_temporal = GATv2GRUForecaster(
        input_dim=256,
        hidden_dim=128,
        dropout=0.2
    ).to(DEVICE)

    e2e_temporal.load_state_dict(
        e2e_ckpt["temporal_state_dict"],
        strict=True
    )

    print("\nGenerating End-to-End predictions...")

    e2e_predictions = generate_predictions(
        e2e_encoder,
        e2e_temporal,
        test_indices,
        Y_test_raw
    )

    e2e_metrics = evaluate_metrics(
        seed,
        "End-to-End",
        e2e_predictions
    )

    all_metrics.extend(e2e_metrics)

    print("End-to-End evaluation completed.")

    # --------------------------------------------
    # C. SAVE NODE-WISE PREDICTIONS
    # --------------------------------------------

    prediction_path = os.path.join(
        OUTPUT_DIR,
        f"predictions_seed_{seed}.npz"
    )

    np.savez_compressed(
        prediction_path,
        target_years=test_years,
        Y_true=Y_test_raw,
        Y_mask=M_test,
        Frozen_predictions=frozen_predictions,
        End_to_End_predictions=e2e_predictions
    )

    print("Predictions saved:", prediction_path)

    # --------------------------------------------
    # MEMORY CLEANUP
    # --------------------------------------------

    del (
        frozen_encoder,
        frozen_temporal,
        e2e_encoder,
        e2e_temporal,
        frozen_predictions,
        e2e_predictions
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ------------------------------------------------
# 12. SAVE DETAILED RESULTS
# ------------------------------------------------

metrics_df = pd.DataFrame(all_metrics)

metrics_path = os.path.join(
    OUTPUT_DIR,
    "test_metrics_by_seed.csv"
)

metrics_df.to_csv(
    metrics_path, index=False
)

print("\n" + "=" * 90)
print("DETAILED TEST RESULTS")
print("=" * 90)

print(
    metrics_df.round(6).to_string(index=False)
)

# ------------------------------------------------
# 13. MEAN AND STANDARD DEVIATION ACROSS SEEDS
# ------------------------------------------------

metric_columns = [
    "MSE", "RMSE", "MAE", "R2"
]

summary_df = (
    metrics_df
    .groupby(
        ["Model", "Year", "Variable"],
        sort=False
    )[metric_columns]
    .agg(["mean", "std"])
    .reset_index()
)

summary_df.columns = [
    "_".join(
        str(x) for x in col if str(x)
    )
    if isinstance(col, tuple)
    else col
    for col in summary_df.columns
]

summary_path = os.path.join(
    OUTPUT_DIR,
    "test_metrics_mean_std.csv"
)

summary_df.to_csv(
    summary_path, index=False
)

print("\n" + "=" * 90)
print("MEAN ± STANDARD DEVIATION ACROSS THREE SEEDS")
print("=" * 90)

print(summary_df.round(6).to_string(index=False))

# ------------------------------------------------
# 14. FINAL COMPLETION
# ------------------------------------------------

print("\n" + "=" * 90)
print("INDEPENDENT TEST EVALUATION COMPLETED")
print("=" * 90)

print("Test years:", test_years.tolist())
print("Seeds:", SEEDS)
print("Output directory:", OUTPUT_DIR)
print("Detailed metrics:", metrics_path)
print("Mean/std summary:", summary_path)

print("\nAll results saved successfully.")


Device: cuda
Seeds: [42, 123, 2023]

Test years: [2024 2025]
Test targets: (2, 5040, 2)
Test masks: (2, 5040, 2)

Training target statistics:
Mean: [ 3.0588877 31.36195  ]
Std : [2.156659 2.990062]

Annual snapshots: 0 to 24
Graph: 5040 nodes

EVALUATING SEED: 42

Generating Frozen predictions...
Frozen evaluation completed.

Generating End-to-End predictions...
End-to-End evaluation completed.
Predictions saved: /kaggle/working/final_test_evaluation/predictions_seed_42.npz

EVALUATING SEED: 123

Generating Frozen predictions...
Frozen evaluation completed.

Generating End-to-End predictions...
End-to-End evaluation completed.
Predictions saved: /kaggle/working/final_test_evaluation/predictions_seed_123.npz

EVALUATING SEED: 2023

Generating Frozen predictions...
Frozen evaluation completed.

Generating End-to-End predictions...
End-to-End evaluation completed.
Predictions saved: /kaggle/working/final_test_evaluation/predictions_seed_2023.npz

DETAILED TEST RESULTS
 Seed      Model    